In [11]:
from pathlib import Path

import flash as fz
import klib
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import yaml
import dataframe_to_image
from sklearn.preprocessing import OrdinalEncoder
from tabulate import tabulate

SyntaxError: '(' was never closed (analysis.py, line 309)

In [ ]:
config_path = Path("src/house_price_prediction/config.yaml")
with config_path.open("r") as file:
    config = yaml.safe_load(file)
# This is for displaying actual values instead of references in yaml
yaml.Dumper.ignore_aliases = lambda *args: True

### Data Assessment and Cleaning

In [ ]:
df = pd.read_csv(config["paths"]["data"]["interim"]["train"])
df_bak = df.copy()

In [ ]:
df.head()

In [ ]:
# Code to generate col_names_renamed dictionary for config.yaml
col_names_renamed_ls = klib.clean_column_names(df).columns
col_names_renamed = dict(zip(df.columns, col_names_renamed_ls))

col_names_renamed

In [ ]:
df.rename(columns=config["col_names_renamed"], inplace=True)
df.columns.tolist()

In [ ]:
def check_duplicates(df):
    if df.duplicated().any():
        return df[df.duplicated(keep=False)]
    return "There are no duplicate data points in the dataset."


check_duplicates(df)

In [ ]:
df.info()

In [ ]:
# Code to extract features based on their type for config.yaml
num_cols, cat_cols, _ = fz.extract_features(df, "all", ignore_cols=["price"])

In [ ]:
cat_cols, numc_cols, numd_cols, target = (
    config["cols"]["cat"],
    config["cols"]["numc"],
    config["cols"]["numd"],
    config["cols"]["target"],
)

In [ ]:
df[cat_cols]

In [ ]:
df[numc_cols]

In [ ]:
df[numd_cols]

In [ ]:
df[[target]]

In [ ]:
headers = ["Feature Name", "Before", "After"]
rows = []
for col in cat_cols:
    before = df[col].unique()
    df[col] = df[col].str.lower().str.replace(" ", "_")
    rows.append([col, list(before), list(df[col].unique())])
print(tabulate(rows, headers, "github"))

In [ ]:
def determine_feature_type(df, col, numc_cols, numd_cols, cat_cols):
    if col in numc_cols:
        return "Continuous Numerical"
    elif col in numd_cols:
        return "Discrete Numerical"
    elif col in cat_cols:
        if len(list(df[col].unique())) == 2:
            return "Binary Categorical"
        elif len(list(df[col].unique())) > 2:
            return "Multi Categorical"
        else:
            return "Single Value Categorical. Maybe remove it?"
    else:
        return "Other"


def get_feature_summary_md(df, numc_cols, numd_cols, cat_cols, exclude_col=None):
    df = df.drop(columns=exclude_col)
    dtype_dict = {
        "bedrooms": {"from": "int64", "to": "category"},
        "bathrooms": {"from": "int64", "to": "category"},
        "neighborhood": {"from": "object", "to": "category"},
    }
    feature_names, feature_types, na_pct, dtypes = [], [], [], []
    for col in list(df.columns):
        feature_names.append(col)
        feature_types.append(
            determine_feature_type(df, col, numc_cols, numd_cols, cat_cols)
        )
        na_pct.append(df[col].isna().sum() / len(df[col]))
        if col in dtype_dict:
            dtypes.append(f"`{dtype_dict[col]['from']}` -> `{dtype_dict[col]['to']}`")
        else:
            dtypes.append(f"`{df[col].dtype}`")
    data = {
        "Feature Name": feature_names,
        "Type": feature_types,
        "Missing (%)": na_pct,
        "Dtype": dtypes,
    }
    return pd.DataFrame(data).to_markdown()

In [ ]:
print(get_feature_summary_md(df, numc_cols, numd_cols, cat_cols, "price"))

In [ ]:
df[cat_cols + numd_cols] = df[cat_cols + numd_cols].astype("category")
df.dtypes

### EDA

#### Univariate Analysis

##### Numerical

In [ ]:
df[numc_cols + [target]].describe().T

In [6]:
test = df[numc_cols + [target]].describe().T
test["skew"] = df[numc_cols + [target]].skew()
test["kurtosis"] = df[numc_cols + [target]].kurtosis()
test["iqr"] = test["75%"] - test["25%"]
test["na_pct"] = df[numc_cols + [target]].isna().sum() / test["count"]

test = test.round(3).map(lambda x: int(x) if x.is_integer() else x)

dataframe_to_image.convert(test,visualisation_library='matplotlib')

# plt.savefig('describe_output.png', bbox_inches='tight')
# plt.close()


NameError: name 'df' is not defined

In [ ]:
test

In [ ]:
fz.stats_moments(df[numc_cols + [target]])

In [ ]:
fig, axs = fz.hist_box_viz(df[numc_cols])
fig

##### Categorical

In [ ]:
df[cat_cols + numd_cols].describe()

In [ ]:
fig, axs = fz.count_viz(df[cat_cols])
fig

### Modeling

### Others

In [ ]:
config.setdefault("cols", {})
config.setdefault("col_names_renamed", {})
config.setdefault("preprocessing", {})
config["cols"]["numc"] = numc_cols
config["cols"]["numd"] = numd_cols
config["cols"]["cat"] = cat_cols
config["cols"]["target"] = target
config["col_names_renamed"] = col_names_renamed
config["preprocessing"]["oe"] = {"neighborhood": ["rural", "suburb", "urban"]}
print(yaml.dump(config))

In [ ]:
with config_path.open("w") as file:
    yaml.dump(config, file)